# 03 - Regional category recovery time (100-mi)

Trend-based recovery time (days from landfall) per category. **Recovery only** (no drop panel).
Plot-only: reads `00`'s `regional_metrics_summary_100mi.csv`.

Three views, each a separate file (matching the existing 50-mi figures):
- `figure3_recovery_within_HvM`  - within recovery per category, Helene vs Milton (cf. figure2d)
- `figure3_recovery_inflow_HvM`  - inflow recovery per category, Helene vs Milton
- `figure3_recovery_withinVSinflow` - within vs inflow per category, one subpanel per storm (cf. figure3a)

In [1]:
import pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')

mpl.rcParams.update({
    'font.family':'sans-serif','font.sans-serif':['Arial','Helvetica','DejaVu Sans'],
    'font.size':8,'axes.titlesize':8,'axes.labelsize':8,'xtick.labelsize':7,'ytick.labelsize':7,
    'legend.fontsize':7,'axes.linewidth':0.6,'axes.spines.top':False,'axes.spines.right':False,
    'savefig.dpi':300,'savefig.bbox':'tight','pdf.fonttype':42,'ps.fonttype':42,
})

In [2]:
# Path resolution -> the 100-mi regional data produced by 00_regional_flows_100mi.ipynb
def find_project_root(start):
    sentinel = pathlib.Path('results')/'npj_100mi'/'regional_data'/'regional_metrics_summary_100mi.csv'
    for p in [start, *start.parents]:
        if (p/sentinel).exists():
            return p
    raise FileNotFoundError(f'Run 00 first; cannot find {sentinel} from {start}')

PROJECT_ROOT = find_project_root(pathlib.Path.cwd())
RESULTS  = PROJECT_ROOT/'results'
REGIONAL = RESULTS/'npj_100mi'/'regional_data'          # 100-mi baselines + metrics (from 00)
METRICS_CSV = REGIONAL/'regional_metrics_summary_100mi.csv'
OUT_DIR  = RESULTS/'npj_100mi'                            # flat figure outputs
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('REGIONAL exists:', REGIONAL.exists(), '| METRICS exists:', METRICS_CSV.exists())

REGIONAL exists: True | METRICS exists: True


In [3]:
HURRICANES = {
    'helene': {'label':'Helene','landing':pd.Timestamp('2024-09-26'),'color':'#1f77b4'},
    'milton': {'label':'Milton','landing':pd.Timestamp('2024-10-09'),'color':'#d62728'},
}
CATEGORIES = ['Travel','Work & Professional','Health','Education','Retail & Leisure','Urban Government']
CATEGORY_COLORS = {'Travel':'#0072B2','Work & Professional':'#E69F00','Health':'#009E73',
    'Education':'#CC79A7','Retail & Leisure':'#56B4E9','Urban Government':'#D55E00'}
FLOW_COLORS = {'within':'#2b8a3e','inflow':'#1971c2','outflow':'#c92a2a'}
FLOW_LABELS = {'within':'Within','inflow':'Inflow','outflow':'Outflow'}

def category_to_filename(c): return c.replace(' & ','_and_').replace(' ','_')
def save_panel(fig, stem):
    fig.savefig(OUT_DIR/f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(OUT_DIR/f'{stem}.png', dpi=300, bbox_inches='tight')
    print('  saved ->', stem, '(.pdf/.png)')

METRICS = pd.read_csv(METRICS_CSV)
METRICS = METRICS[METRICS['category'].isin(CATEGORIES)].copy()
print('metrics rows (6 categories):', len(METRICS))

metrics rows (6 categories): 36


In [4]:
# Helene-vs-Milton grouped bars (reused from figure2_categories), recovery only
def metrics_pivot(metric_col, flow):
    sub = METRICS[METRICS['flow_type']==flow]
    return sub.pivot(index='category', columns='hurricane', values=metric_col).reindex(CATEGORIES)[list(HURRICANES)]

def grouped_bar(ax, piv, ylabel):
    cats=list(piv.index); x=np.arange(len(cats)); width=0.38
    off={'helene':-width/2,'milton':+width/2}
    for hkey in HURRICANES:
        vals=piv[hkey].values
        bars=ax.bar(x+off[hkey], vals, width, color=HURRICANES[hkey]['color'],
                    edgecolor='white', linewidth=0.4, label=HURRICANES[hkey]['label'])
        for r,v in zip(bars,vals):
            if not np.isnan(v):
                ax.text(r.get_x()+r.get_width()/2, v+0.1, f'{v:.1f}', ha='center', va='bottom', fontsize=6.2, color='#333')
    ax.set_xticks(x); ax.set_xticklabels(cats, rotation=20, ha='right')
    for tick,cat in zip(ax.get_xticklabels(),cats): tick.set_color(CATEGORY_COLORS[cat])
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, np.nanmax(piv.values)*1.3)                  # headroom for the legend row
    ax.legend(loc='upper right', frameon=False, ncol=2, columnspacing=1.0, handletextpad=0.4)

for flow in ['within','inflow']:
    piv = metrics_pivot('recovery_days', flow)
    print(f'\n{flow} recovery (days):'); print(piv.round(2))
    fig, ax = plt.subplots(figsize=(3.6,2.0))
    grouped_bar(ax, piv, 'Recovery time (days from landfall)')
    ax.set_title(f'{FLOW_LABELS[flow]}-region recovery (Theil-Sen, 100 mi)', loc='left', fontsize=8, color='#333', pad=4)
    save_panel(fig, f'figure3_recovery_{flow}_HvM'); plt.close(fig)


within recovery (days):
hurricane            helene  milton
category                           
Travel                 5.09    4.75
Work & Professional    4.61    4.91
Health                 4.49    4.75
Education              3.21    6.05
Retail & Leisure       5.07    5.20
Urban Government       4.72    5.01


  saved -> figure3_recovery_within_HvM (.pdf/.png)

inflow recovery (days):
hurricane            helene  milton
category                           
Travel                13.31    5.03
Work & Professional   11.79    5.24
Health                12.15    4.61
Education             11.30    4.38
Retail & Leisure      11.90    5.01
Urban Government      12.51    5.49


  saved -> figure3_recovery_inflow_HvM (.pdf/.png)


In [5]:
# Within vs inflow recovery, one subpanel per storm (reused from figure3_flow_decomposition panel a)
def _wrap(c):
    if ' & ' in c: return c.replace(' & ',' &\n')
    parts=c.split(' ',1); return '\n'.join(parts) if len(parts)==2 else c

fig, axes = plt.subplots(1, 2, figsize=(7.4,3.4))
x=np.arange(len(CATEGORIES)); bw=0.38
for ax,hkey in zip(axes, HURRICANES):
    sub=METRICS[(METRICS['hurricane']==hkey)&(METRICS['flow_type'].isin(['within','inflow']))]
    piv=sub.pivot_table(index='category', columns='flow_type', values='recovery_days', aggfunc='first').reindex(CATEGORIES)
    ax.bar(x-bw/2, piv['within'].values, bw, color=FLOW_COLORS['within'], edgecolor='white', linewidth=0.5, label='Within')
    ax.bar(x+bw/2, piv['inflow'].values, bw, color=FLOW_COLORS['inflow'], edgecolor='white', linewidth=0.5, label='Inflow')
    for xi,(w,iv) in enumerate(zip(piv['within'].values, piv['inflow'].values)):
        if pd.notna(w):  ax.text(xi-bw/2, w+0.15, f'{w:.1f}', ha='center', va='bottom', fontsize=6, color=FLOW_COLORS['within'])
        if pd.notna(iv): ax.text(xi+bw/2, iv+0.15, f'{iv:.1f}', ha='center', va='bottom', fontsize=6, color=FLOW_COLORS['inflow'])
    ax.set_xticks(x); ax.set_xticklabels([_wrap(c) for c in CATEGORIES], rotation=0, ha='center')
    ax.set_ylabel('Recovery time (days from landing)')
    ax.set_title(HURRICANES[hkey]['label'], loc='left', fontsize=9, color=HURRICANES[hkey]['color'], pad=2)
    ax.grid(axis='y', linestyle=':', color='#bbb', linewidth=0.5, alpha=0.7); ax.set_axisbelow(True)
ymax=max(a.get_ylim()[1] for a in axes)
for a in axes: a.set_ylim(0, ymax)
axes[-1].legend(loc='upper right', frameon=False, fontsize=7, ncol=2)
fig.tight_layout()
save_panel(fig, 'figure3_recovery_withinVSinflow'); plt.close(fig)
print('done: 3 recovery panels')

  saved -> figure3_recovery_withinVSinflow (.pdf/.png)
done: 3 recovery panels
